# SysML v2 LSP — Python Notebook Demo

This notebook demonstrates how to drive the **SysML v2 Language Server** from Python,
using the exact same JSON-RPC/stdio protocol that VS Code uses.

No third-party packages are required — only the Python standard library.

## Prerequisites

1. **Node.js >= 20** installed
2. The LSP server bundle built at `dist/server/server.js`:
   ```bash
   cd ../..   # repository root
   npm install && npm run build
   ```

## Architecture

```
┌────────────────────┐     stdio (JSON-RPC)      ┌────────────────────┐
│  This Notebook     │ ◄──────────────────────►  │  server.js (Node)  │
│  (Python 3.10+)    │  Content-Length framing   │  ANTLR4 parser     │
└────────────────────┘                           └────────────────────┘
```

## 1. Setup — Import the LSP Client

We import the lightweight `JsonRpcClient` and helper functions from the companion script `sysml_lsp_client.py` (in this same directory). It handles Content-Length framing, request/response matching, and notification draining.

In [2]:
import sys
import json
from pathlib import Path

# Ensure the client module is importable
client_dir = Path.cwd() if Path("sysml_lsp_client.py").exists() else Path("clients/python")
if str(client_dir) not in sys.path:
    sys.path.insert(0, str(client_dir.resolve()))

from sysml_lsp_client import (
    JsonRpcClient,
    start_server,
    initialize,
    open_document,
    close_document,
    reload_document,
    get_document_symbols,
    get_hover,
    get_completions,
    get_definition,
    get_folding_ranges,
    shutdown,
    file_uri,
    SYMBOL_KIND,
    SEVERITY,
)

print("LSP client module loaded successfully.")

LSP client module loaded successfully.


## 2. Resolve Paths

We locate the repository root and the pre-built server bundle.

In [3]:
# Find the repository root (works whether the notebook is run from clients/python or the repo root)
notebook_dir = Path.cwd()
if (notebook_dir / "dist" / "server" / "server.js").exists():
    repo_root = notebook_dir
elif (notebook_dir.parent.parent / "dist" / "server" / "server.js").exists():
    repo_root = notebook_dir.parent.parent
else:
    raise FileNotFoundError(
        "Cannot find dist/server/server.js — run 'npm run build' from the repo root first."
    )

server_js = repo_root / "dist" / "server" / "server.js"
examples_dir = repo_root / "examples"

print(f"Repo root  : {repo_root}")
print(f"Server JS  : {server_js}")
print(f"Examples   : {examples_dir}")
print(f"SysML files: {[p.name for p in sorted(examples_dir.glob('*.sysml'))]}")

Repo root  : /workspaces/sysml-v2-lsp
Server JS  : /workspaces/sysml-v2-lsp/dist/server/server.js
Examples   : /workspaces/sysml-v2-lsp/examples
SysML files: ['bike.sysml', 'camera.sysml', 'toaster-system.sysml', 'vehicle-model.sysml']


## 3. Start the Language Server

We spawn the Node.js language server process and perform the LSP `initialize` / `initialized` handshake.

In [4]:
# Launch the server as a subprocess communicating over stdio
proc = start_server(server_js)
client = JsonRpcClient(proc)

# Perform the LSP handshake
init_result = initialize(client, repo_root)

capabilities = init_result.get("result", {}).get("capabilities", {})
cap_names = sorted(k for k, v in capabilities.items() if v)

print("Server initialized successfully!")
print(f"\nCapabilities ({len(cap_names)}):")
for name in cap_names:
    print(f"  - {name}")

Server initialized successfully!

Capabilities (9):
  - completionProvider
  - definitionProvider
  - documentSymbolProvider
  - foldingRangeProvider
  - hoverProvider
  - referencesProvider
  - renameProvider
  - semanticTokensProvider
  - textDocumentSync


## 4. Open a SysML Document

We send `textDocument/didOpen` to tell the server about a `.sysml` file, then wait for it to parse the file and publish diagnostics.

> **Re-run safe:** If the document is already open, `open_document` automatically closes and re-opens it so the server picks up the latest content from disk.

In [5]:
# Pick a file to analyse
sysml_file = examples_dir / "bike.sysml"
print(f"Opening: {sysml_file.relative_to(repo_root)}")

uri = open_document(client, sysml_file)
print(f"URI: {uri}")

# Wait for the server to finish parsing (first file may take a few seconds for DFA warm-up)
print("\nWaiting for server to parse...")
client.drain_until_diagnostics(uri, timeout=30.0)
print("Done!")

Opening: examples/bike.sysml
URI: file:///workspaces/sysml-v2-lsp/examples/bike.sysml

Waiting for server to parse...
Done!


## 5. Document Symbols (Outline)

The `textDocument/documentSymbol` request returns the hierarchical symbol tree — packages, parts, attributes, constraints, etc.

In [6]:
symbols = get_document_symbols(client, uri)

def format_symbol_tree(symbols, indent=0):
    """Recursively format the symbol tree as a readable string."""
    lines = []
    for sym in symbols:
        kind = SYMBOL_KIND.get(sym.get("kind", 0), "?")
        name = sym.get("name", "?")
        line = sym.get("range", {}).get("start", {}).get("line", 0) + 1
        lines.append(f"{'  ' * indent}{kind:12s} {name}  (line {line})")
        children = sym.get("children", [])
        if children:
            lines.extend(format_symbol_tree(children, indent + 1).splitlines())
    return "\n".join(lines)

print(f"Found {len(symbols)} top-level symbol(s):\n")
print(format_symbol_tree(symbols))

Found 1 top-level symbol(s):

Package      Bike  (line 1)
  Property     Mass  (line 4)
  Property     Length  (line 5)
  Property     Color  (line 6)
  Class        Notdd  (line 8)
  Enum         BikeType  (line 11)
  Enum         FrameMaterial  (line 18)
  Interface    MechanicalPort  (line 26)
    Property     torque  (line 27)
  Interface    HydraulicPort  (line 30)
    Property     pressure  (line 31)
  Interface    ChainDrive  (line 35)
  Interface    BrakeCable  (line 40)
  Class        Frame  (line 46)
    Property     material  (line 47)
    Property     frameSize  (line 48)
    Property     frameColor  (line 49)
    Class        seatTube  (line 51)
    Class        downTube  (line 52)
    Class        topTube  (line 53)
    Class        headTube  (line 54)
    Class        chainstay  (line 55)
    Class        seatstay  (line 56)
  Class        Tube  (line 59)
    Property     length  (line 60)
    Property     diameter  (line 61)
  Class        Wheel  (line 64)
    Property 

## 6. Diagnostics (Errors & Warnings)

The server pushes diagnostics via `textDocument/publishDiagnostics` notifications. These are the same errors/warnings you see in VS Code's "Problems" panel.

> **Tip:** If you've edited the `.sysml` file on disk, re-run this cell — it reloads the file so the server re-parses the latest version.

In [8]:
# Reload the file from disk so we always get fresh diagnostics
uri = reload_document(client, sysml_file)
client.drain_until_diagnostics(uri, timeout=30.0)

diagnostics = client.get_diagnostics(uri)

if not diagnostics:
    print("No diagnostics — the file is clean!")
else:
    print(f"{len(diagnostics)} diagnostic(s):\n")
    for d in diagnostics:
        severity = SEVERITY.get(d.get("severity", 1), "?")
        line = d["range"]["start"]["line"] + 1
        col  = d["range"]["start"]["character"] + 1
        msg  = d.get("message", "")
        print(f"  [{severity}] line {line}:{col} — {msg}")

46 diagnostic(s):

  [Info] line 4:19 — Definition 'Mass' has no documentation
  [Info] line 5:19 — Definition 'Length' has no documentation
  [Info] line 6:19 — Definition 'Color' has no documentation
  [Info] line 8:14 — Enumeration 'BikeType' has no enum values defined
  [Info] line 8:14 — Definition 'BikeType' has no documentation
  [Info] line 15:14 — Enumeration 'FrameMaterial' has no enum values defined
  [Info] line 15:14 — Definition 'FrameMaterial' has no documentation
  [Info] line 23:14 — Definition 'MechanicalPort' has no documentation
  [Info] line 27:14 — Definition 'HydraulicPort' has no documentation
  [Info] line 32:19 — Definition 'ChainDrive' has no documentation
  [Info] line 37:19 — Definition 'BrakeCable' has no documentation
  [Info] line 43:14 — Definition 'Frame' has no documentation
  [Info] line 56:14 — Definition 'Tube' has no documentation
  [Info] line 61:14 — Definition 'Wheel' has no documentation
  [Info] line 71:14 — Definition 'Rim' has no documentat

## 7. Hover Information

The `textDocument/hover` request returns documentation/type info when you hover over a symbol — just like in VS Code.

In [9]:
# Hover on the first top-level symbol
if symbols:
    first_sym = symbols[0]
    sel_range = first_sym.get("selectionRange", first_sym.get("range", {}))
    start = sel_range.get("start", {"line": 0, "character": 0})

    print(f"Hovering on '{first_sym['name']}' at line {start['line'] + 1}, col {start['character'] + 1}:\n")

    hover = get_hover(client, uri, start["line"], start["character"])
    if hover:
        contents = hover.get("contents", {})
        if isinstance(contents, dict):
            print(contents.get("value", str(contents)))
        elif isinstance(contents, list):
            for c in contents:
                print(c.get("value", str(c)) if isinstance(c, dict) else str(c))
        else:
            print(str(contents))
    else:
        print("(no hover info returned)")
else:
    print("No symbols found to hover on.")

Hovering on 'Bike' at line 1, col 9:

**package** `Bike`

---
The bicycle shall weigh no more than 12 kg.


## 8. Code Completions

The `textDocument/completion` request returns what the language server would suggest at a given cursor position.

In [10]:
# Request completions at the beginning of the file (line 0, col 0)
completions = get_completions(client, uri, 0, 0)

print(f"Completions at line 1, col 1: {len(completions)} item(s)\n")
for item in completions[:15]:
    label = item.get("label", "?")
    kind  = item.get("kind", "")
    detail = item.get("detail", "")
    print(f"  {label:20s}  kind={kind}  {detail}")

if len(completions) > 15:
    print(f"  ... and {len(completions) - 15} more")

Completions at line 1, col 1: 41 item(s)

  part def              kind=7  Part definition
  action def            kind=3  Action definition
  state def             kind=13  State definition
  requirement def       kind=8  Requirement definition
  constraint def        kind=21  Constraint definition
  attribute def         kind=10  Attribute definition
  item def              kind=7  Item definition
  port def              kind=8  Port definition
  interface def         kind=8  Interface definition
  connection def        kind=8  Connection definition
  allocation def        kind=8  Allocation definition
  use case def          kind=23  Use case definition
  view def              kind=22  View definition
  viewpoint def         kind=22  Viewpoint definition
  enum def              kind=13  Enumeration definition
  ... and 26 more


## 9. Go-to-Definition

The `textDocument/definition` request resolves where a symbol is defined — useful for navigating large models.

In [11]:
# Try go-to-definition on the first symbol
if symbols:
    target = symbols[0]
    sel = target.get("selectionRange", target.get("range", {}))
    pos = sel.get("start", {"line": 0, "character": 0})

    print(f"Go-to-Definition on '{target['name']}' at line {pos['line'] + 1}:\n")

    definition = get_definition(client, uri, pos["line"], pos["character"])
    if definition:
        defs = definition if isinstance(definition, list) else [definition]
        for d in defs:
            def_uri = d.get("uri", "?")
            def_line = d.get("range", {}).get("start", {}).get("line", 0) + 1
            print(f"  {def_uri}  line {def_line}")
    else:
        print("  (no definition found — may be a top-level declaration)")
else:
    print("No symbols to look up.")

Go-to-Definition on 'Bike' at line 1:

  file:///workspaces/sysml-v2-lsp/examples/bike.sysml  line 1


## 10. Folding Ranges

The server reports which regions of code can be collapsed — useful for understanding document structure.

In [11]:
folds = get_folding_ranges(client, uri)

print(f"{len(folds)} foldable region(s):\n")
for f in folds[:10]:
    start_line = f.get("startLine", 0) + 1
    end_line   = f.get("endLine", 0) + 1
    kind       = f.get("kind", "region")
    print(f"  lines {start_line:3d} — {end_line:3d}  ({kind})")

if len(folds) > 10:
    print(f"  ... and {len(folds) - 10} more")

39 foldable region(s):

  lines   8 —  13  (region)
  lines  15 —  20  (region)
  lines  23 —  25  (region)
  lines  27 —  29  (region)
  lines  32 —  35  (region)
  lines  37 —  40  (region)
  lines  43 —  54  (region)
  lines  56 —  59  (region)
  lines  61 —  69  (region)
  lines  71 —  73  (region)
  ... and 29 more


## 11. Analyse Multiple Files

Since the server stays running, we can open additional documents and query them without restarting.

In [12]:
sysml_files = sorted(examples_dir.glob("*.sysml"))

summary = []
for f in sysml_files:
    if f == sysml_file:
        # Already opened above
        syms = symbols
        diags = diagnostics
    else:
        f_uri = open_document(client, f)
        client.drain_until_diagnostics(f_uri, timeout=30.0)
        syms = get_document_symbols(client, f_uri)
        diags = client.get_diagnostics(f_uri)

    def count_symbols(sym_list):
        total = len(sym_list)
        for s in sym_list:
            total += count_symbols(s.get("children", []))
        return total

    summary.append({
        "file": f.name,
        "symbols": count_symbols(syms),
        "errors": sum(1 for d in diags if d.get("severity") == 1),
        "warnings": sum(1 for d in diags if d.get("severity") == 2),
    })

# Print a summary table
print(f"{'File':<35s} {'Symbols':>8s} {'Errors':>7s} {'Warnings':>9s}")
print("-" * 62)
for row in summary:
    print(f"{row['file']:<35s} {row['symbols']:>8d} {row['errors']:>7d} {row['warnings']:>9d}")
print("-" * 62)
print(f"{'TOTAL':<35s} {sum(r['symbols'] for r in summary):>8d} "
      f"{sum(r['errors'] for r in summary):>7d} {sum(r['warnings'] for r in summary):>9d}")

File                                 Symbols  Errors  Warnings
--------------------------------------------------------------
bike.sysml                               126       0         3
camera.sysml                               6       0         3
toaster-system.sysml                      68       0         0
vehicle-model.sysml                       42       0         0
--------------------------------------------------------------
TOTAL                                    242       0         6


## 12. Analyse Inline SysML Text

You don't need a file on disk — you can send arbitrary SysML text to the server using `textDocument/didOpen` with a synthetic URI. This makes the notebook a great scratchpad for experimenting with SysML models.

In [13]:
# Define some SysML inline — no file needed
inline_sysml = """\
package RobotArm {
    part def Joint {
        attribute angle : ScalarValues::Real;
        attribute maxTorque : ScalarValues::Real;
    }

    part def Segment {
        attribute length : ScalarValues::Real;
    }

    part def Arm {
        part shoulder : Joint;
        part elbow : Joint;
        part wrist : Joint;

        part upperArm : Segment;
        part forearm : Segment;
    }
}
"""

# Send it to the server with a synthetic URI
inline_uri = "file:///virtual/scratchpad.sysml"
client.notify("textDocument/didOpen", {
    "textDocument": {
        "uri": inline_uri,
        "languageId": "sysml",
        "version": 1,
        "text": inline_sysml,
    },
})

# Wait for diagnostics
client.drain_until_diagnostics(inline_uri, timeout=15.0)

# Inspect the inline model
inline_symbols = get_document_symbols(client, inline_uri)
inline_diags = client.get_diagnostics(inline_uri)

print("Inline SysML Model — Symbol Tree:\n")
print(format_symbol_tree(inline_symbols))

print(f"\nDiagnostics: ", end="")
if not inline_diags:
    print("clean!")
else:
    print(f"{len(inline_diags)} issue(s)")
    for d in inline_diags:
        sev = SEVERITY.get(d.get("severity", 1), "?")
        ln = d["range"]["start"]["line"] + 1
        print(f"  [{sev}] line {ln}: {d.get('message', '')}")

Inline SysML Model — Symbol Tree:

Package      RobotArm  (line 1)
  Class        Joint  (line 2)
    Property     angle  (line 3)
    Property     maxTorque  (line 4)
  Class        Segment  (line 7)
    Property     length  (line 8)
  Class        Arm  (line 11)
    Class        shoulder  (line 12)
    Class        elbow  (line 13)
    Class        wrist  (line 14)
    Class        upperArm  (line 16)
    Class        forearm  (line 17)

Diagnostics: clean!


## 13. Shutdown the Server

Always perform a clean `shutdown` / `exit` handshake to release the server process.

In [13]:
shutdown(client)
proc.wait(timeout=5)

print("Server shut down cleanly.")
print("\nThe same LSP that powers VS Code was driven entirely from a Python notebook!")

Server shut down cleanly.

The same LSP that powers VS Code was driven entirely from a Python notebook!


---

## What Next?

Now that you can drive the LSP from Python, consider:

- **CI validation** — parse all `.sysml` files and fail on diagnostics
- **Model documentation** — extract symbols and auto-generate docs
- **Data analysis** — count parts, constraints, etc. across a model set
- **Interactive editing** — send `textDocument/didChange` to experiment with modifications
- **Custom tooling** — combine LSP queries with pandas, matplotlib, or other Python libraries